# Proyecto Final — Airbnb: Madrid, Mallorca, Valencia, Sevilla, Milán
## Notebook 1: Limpieza y Transformación de Datos

**Objetivo:** Construir el dataset final limpio y enriquecido a partir de las fuentes originales.

### Fuentes de datos
| Archivo | Fuente | Descripción |
|---|---|---|
| `madrid_listings.csv` | Inside Airbnb | Anuncios Airbnb Madrid (sep. 2025) |
| `mallorca_listings.csv` | Inside Airbnb | Anuncios Airbnb Mallorca |
| `valencia_listings.csv` | Inside Airbnb | Anuncios Airbnb Valencia |
| `sevilla_listings.csv` | Inside Airbnb | Anuncios Airbnb Sevilla |
| `milan_listings.csv` | Inside Airbnb | Anuncios Airbnb Milán (dic. 2025) |
| `turismo.csv` | Eurostat (`tour_occ_nin3`) | Pernoctaciones en alojamientos turísticos por región NUTS3 |

### Pipeline de transformación
```
data/raw/*.csv  (intocables)
    │
    ├── PASO 1: Limpieza individual por ciudad
    │           → data/processed/*_clean.csv  (27 cols, mismas en todas)
    │
    ├── PASO 2: Concatenación + merge con Eurostat
    │           → data/processed/listings_merged.csv  (30 cols, más crudo)
    │
    └── PASO 3: Limpieza global + feature engineering
                → data/processed/listings_final.csv  (28 cols, listo para análisis)
```

> **Principio de integridad:** Los archivos en `data/raw/` son intocables. Todas las transformaciones se realizan en Python y se documentan en este notebook.

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

RAW  = '../data/raw'
PROC = '../data/processed'
os.makedirs(PROC, exist_ok=True)

pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_columns', None)
print('Entorno listo')

## 1. Carga de datos en bruto

In [ ]:
archivos = {
    'Madrid':   'madrid_listings.csv',
    'Mallorca': 'mallorca_listings.csv',
    'Valencia': 'valencia_listings.csv',
    'Sevilla':  'sevilla_listings.csv',
    'Milan':    'milan_listings.csv',
}

print(f'{"Ciudad":<12} {"País":<8} {"Filas":>8} {"Cols":>6} {"Price nulos":>13}')
print('-' * 52)
paises = {'Madrid':'España','Mallorca':'España','Valencia':'España','Sevilla':'España','Milan':'Italia'}
for ciudad, archivo in archivos.items():
    df_tmp = pd.read_csv(f'{RAW}/{archivo}', low_memory=False)
    print(f'{ciudad:<12} {paises[ciudad]:<8} {len(df_tmp):>8,} '
          f'{df_tmp.shape[1]:>6} {df_tmp["price"].isna().sum():>13,}')

eur_raw = pd.read_csv(f'{RAW}/turismo.csv', low_memory=False)
print(f'\nEurostat (turismo): {len(eur_raw):,} filas x {eur_raw.shape[1]} columnas')

## PASO 1: Limpieza individual por ciudad → `*_clean.csv`

Cada ciudad se limpia de forma **idéntica** para garantizar coherencia:

| Transformación | Detalle |
|---|---|
| Columnas basura | Eliminación de URLs, textos libres, thumbnails |
| `price` | Eliminar `$` y `,` → float; eliminar nulos y ceros |
| Tipos numéricos | `bathrooms`, `beds`, `bedrooms` → float |
| Booleanos | `host_is_superhost`, `instant_bookable`: `'t'/'f'` → `True/False` |
| Nulos básicos | `reviews_per_month` → 0; `review_scores_*` → mediana por ciudad |
| Fechas | `host_since`, `first_review`, `last_review` → datetime |
| Features básicas | `price_per_person`, `high_availability` |
| Columnas finales | 27 columnas coherentes entre todas las ciudades |

In [ ]:
COLS_DROP = [
    'listing_url','picture_url','host_url','host_about','description',
    'amenities','host_thumbnail_url','host_picture_url',
    'neighborhood_overview','scrape_id','calendar_updated',
    'calendar_last_scraped','host_verifications',
    'host_profile_url','host_profile_id'
]

COLS_CLEAN = [
    'city','neighbourhood_cleansed','neighbourhood_group_cleansed',
    'latitude','longitude','room_type','property_type',
    'accommodates','bathrooms','bedrooms','beds',
    'price','price_per_person','high_availability',
    'minimum_nights','maximum_nights','availability_365',
    'number_of_reviews','review_scores_rating','review_scores_location',
    'reviews_per_month','host_is_superhost','instant_bookable',
    'calculated_host_listings_count','host_since','first_review','last_review'
]

def limpiar_ciudad(path, ciudad):
    df = pd.read_csv(path, low_memory=False)
    df['city'] = ciudad
    n_inicial = len(df)

    df = df.drop_duplicates()
    n_dup = n_inicial - len(df)

    df.drop(columns=[c for c in COLS_DROP if c in df.columns], inplace=True)

    df['price'] = (df['price'].astype(str)
                   .str.replace(r'[$,]', '', regex=True)
                   .str.strip().replace('nan', np.nan).astype(float))
    n_sin_precio = (df['price'].isna() | (df['price'] <= 0)).sum()
    df = df[df['price'].notna() & (df['price'] > 0)].copy()

    for col in ['bathrooms','beds','bedrooms']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df[col] = df[col].fillna(df[col].median())

    df['host_is_superhost'] = df['host_is_superhost'] == 't'
    df['instant_bookable']  = df['instant_bookable']  == 't'

    df['reviews_per_month']      = df['reviews_per_month'].fillna(0)
    df['review_scores_rating']   = df['review_scores_rating'].fillna(df['review_scores_rating'].median())
    df['review_scores_location'] = df['review_scores_location'].fillna(df['review_scores_location'].median())

    df['room_type']     = df['room_type'].astype(str).str.strip()
    df['property_type'] = df['property_type'].astype(str).str.strip()

    for col in ['host_since','first_review','last_review']:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')

    df['price_per_person']  = (df['price'] / df['accommodates'].replace(0, np.nan)).round(2)
    df['high_availability'] = df['availability_365'] > 180

    cols = [c for c in COLS_CLEAN if c in df.columns]
    df = df[cols].copy()

    print(f'  {ciudad:<10} | {n_inicial:>6,} → {len(df):>6,} filas '
          f'| dup: {n_dup} | sin precio: {n_sin_precio} | {df.shape[1]} cols')
    return df

print('PASO 1 — Limpieza por ciudad')
print()
mad = limpiar_ciudad(f'{RAW}/madrid_listings.csv',   'Madrid')
val = limpiar_ciudad(f'{RAW}/valencia_listings.csv', 'Valencia')
mal = limpiar_ciudad(f'{RAW}/mallorca_listings.csv', 'Mallorca')
sev = limpiar_ciudad(f'{RAW}/sevilla_listings.csv',  'Sevilla')
mil = limpiar_ciudad(f'{RAW}/milan_listings.csv',    'Milan')

In [ ]:
# Verificar coherencia de columnas entre ciudades
sets = [set(d.columns) for d in [mad, val, mal, sev, mil]]
comunes = set.intersection(*sets)
print(f'Columnas comunes entre todas las ciudades: {len(comunes)} ✅')
for nombre, df_c in [('Madrid',mad),('Valencia',val),('Mallorca',mal),('Sevilla',sev),('Milan',mil)]:
    print(f'  {nombre:<10}: {df_c.shape[1]} cols | dtypes: {dict(df_c.dtypes.value_counts())}')

In [ ]:
mad.to_csv(f'{PROC}/madrid_clean.csv',   index=False)
val.to_csv(f'{PROC}/valencia_clean.csv', index=False)
mal.to_csv(f'{PROC}/mallorca_clean.csv', index=False)
sev.to_csv(f'{PROC}/sevilla_clean.csv',  index=False)
mil.to_csv(f'{PROC}/milan_clean.csv',    index=False)
print('Archivos *_clean.csv guardados en data/processed/')

## PASO 2: Concatenación + merge con Eurostat → `listings_merged.csv`

> **Nota sobre Milán y los datos de turismo:**  
> Los datos de pernoctaciones proceden de Eurostat (`tour_occ_nin3`), que sí incluye la región **Milano** (ISTAT/Eurostat). Por tanto, Milán **tiene datos de turismo disponibles** y `pernoctaciones_total` y `tourism_pressure` son válidos para las 5 ciudades.  
> La distinción relevante es que las 4 ciudades españolas comparten el mismo marco estadístico del Eurostat-España, mientras que Milán usa la fuente italiana (ISTAT). Esto se señala en el informe cuando se hacen comparativas de contexto turístico.

In [ ]:
df_merged = pd.concat([mad, val, mal, sev, mil], ignore_index=True)
print(f'Concat: {df_merged.shape[0]:,} filas x {df_merged.shape[1]} columnas')
print(df_merged['city'].value_counts())

In [ ]:
eur = pd.read_csv(f'{RAW}/turismo.csv', low_memory=False)

eur_f = eur[
    (eur['geo'].isin(['Madrid','Valencia/València','Mallorca','Sevilla','Milano'])) &
    (eur['unit'] == 'Number') &
    (eur['c_resid'].isin(['Domestic country','Foreign country'])) &
    (eur['nace_r2'].str.startswith('Hotels; holiday'))
].copy()

eur_f['tipo'] = eur_f['c_resid'].map({
    'Domestic country': 'pernoctaciones_nacionales',
    'Foreign country':  'pernoctaciones_internacionales'
})

eur_wide = (
    eur_f.sort_values('TIME_PERIOD', ascending=False)
         .groupby(['geo','tipo']).first().reset_index()
         [['geo','tipo','OBS_VALUE']]
         .pivot(index='geo', columns='tipo', values='OBS_VALUE')
         .reset_index().rename(columns={'geo':'city_eur'})
)
eur_wide.columns.name = None
eur_wide['pernoctaciones_total'] = (
    eur_wide['pernoctaciones_nacionales'] + eur_wide['pernoctaciones_internacionales']
)
print('Eurostat preparado — pernoctaciones_total por región:')
print(eur_wide[['city_eur','pernoctaciones_total']].round(0).to_string(index=False))

In [ ]:
MAPA = {'Madrid':'Madrid','Valencia':'Valencia/València',
        'Mallorca':'Mallorca','Sevilla':'Sevilla','Milan':'Milano'}
df_merged['city_eur'] = df_merged['city'].map(MAPA)
df_merged = df_merged.merge(eur_wide, on='city_eur', how='left').drop(columns='city_eur')

print(f'listings_merged: {df_merged.shape[0]:,} filas x {df_merged.shape[1]} columnas')
print('\nVerificación — pernoctaciones_total por ciudad:')
print(df_merged.groupby('city')['pernoctaciones_total'].first().sort_values(ascending=False).round(0))
print(f'\nNulos en pernoctaciones_total: {df_merged["pernoctaciones_total"].isna().sum()}')

df_merged.to_csv(f'{PROC}/listings_merged.csv', index=False)
print('\nlistings_merged.csv guardado')

## PASO 3: Limpieza global + feature engineering → `listings_final.csv`

| Operación | Dónde | Justificación |
|---|---|---|
| Duplicados | final | Pueden aparecer tras el concat |
| Outliers precio p99 por ciudad | final | Cada ciudad tiene distribución diferente |
| Cap 2.000€ Mallorca | final | Valores 8k-10k son precios mensuales mal etiquetados |
| Imputación global `review_scores_*` | final | Coherencia entre ciudades |
| `tourism_pressure` | final | Requiere el total de pernoctaciones de todas las ciudades |
| `is_popular`, `precio_categoria` | final | Percentiles globales, no por ciudad |
| `host_antiguedad_años` | final | Requiere todas las fechas cargadas |
| Selección columnas | final | 30 → 28 columnas analíticamente relevantes |

In [ ]:
df_final = df_merged.copy()

# 1. Duplicados
antes = len(df_final)
df_final = df_final.drop_duplicates()
print(f'Duplicados eliminados: {antes - len(df_final)}')

# 2. Outliers precio por ciudad
umbrales = {
    'Madrid':   df_final[df_final['city']=='Madrid']['price'].quantile(0.99),
    'Valencia': df_final[df_final['city']=='Valencia']['price'].quantile(0.99),
    'Sevilla':  df_final[df_final['city']=='Sevilla']['price'].quantile(0.99),
    'Milan':    df_final[df_final['city']=='Milan']['price'].quantile(0.99),
    'Mallorca': 2000.0
}
print('\nUmbrales precio por ciudad:')
for c, u in umbrales.items():
    print(f'  {c:<12}: {u:.0f}€')

df_final['_umbral'] = df_final['city'].map(umbrales)
antes = len(df_final)
df_final = df_final[df_final['price'] <= df_final['_umbral']].drop(columns='_umbral').reset_index(drop=True)
print(f'\nOutliers precio eliminados: {antes - len(df_final)}')

In [ ]:
# 3. Imputación global
for col in ['review_scores_rating','review_scores_location']:
    nulos = df_final[col].isna().sum()
    if nulos > 0:
        df_final[col] = df_final[col].fillna(df_final[col].median())
        print(f'{col}: {nulos} nulos → mediana global')
df_final['reviews_per_month'] = df_final['reviews_per_month'].fillna(0)

In [ ]:
# 4. Features globales
df_final['tourism_pressure'] = (
    df_final['pernoctaciones_total'] / df_final['pernoctaciones_total'].max()).round(4)
df_final['is_popular']       = df_final['number_of_reviews'] > df_final['number_of_reviews'].median()
df_final['host_profesional'] = df_final['calculated_host_listings_count'] > 3
df_final['host_since']       = pd.to_datetime(df_final['host_since'], errors='coerce')
df_final['host_antiguedad_años'] = (
    (pd.Timestamp('2025-12-01') - df_final['host_since']).dt.days / 365.25).round(1)
df_final['precio_categoria'] = pd.cut(
    df_final['price'],
    bins=[0, df_final['price'].quantile(0.25), df_final['price'].quantile(0.50),
          df_final['price'].quantile(0.75), df_final['price'].max()],
    labels=['Económico','Medio-bajo','Medio-alto','Premium'], include_lowest=True)

print('Features globales creadas:')
for f in ['tourism_pressure','is_popular','host_profesional','host_antiguedad_años','precio_categoria']:
    print(f'  ✅ {f}')

In [ ]:
# 5. Selección columnas finales
COLS_FINAL = [
    'city','neighbourhood_cleansed',
    'latitude','longitude','room_type','property_type',
    'accommodates','bathrooms','bedrooms','beds',
    'price','price_per_person','precio_categoria',
    'minimum_nights','availability_365','high_availability',
    'number_of_reviews','review_scores_rating','review_scores_location',
    'reviews_per_month','is_popular',
    'host_is_superhost','host_antiguedad_años','host_profesional',
    'calculated_host_listings_count','instant_bookable',
    'pernoctaciones_total','tourism_pressure'
]
df_final = df_final[[c for c in COLS_FINAL if c in df_final.columns]].copy()

## Verificación final

In [ ]:
print('=' * 55)
print('RESUMEN DEL DATASET FINAL')
print('=' * 55)
print(f'  Filas:        {df_final.shape[0]:,}')
print(f'  Columnas:     {df_final.shape[1]}')
print(f'  Duplicados:   {df_final.duplicated().sum()}')
print(f'  Price nulos:  {df_final["price"].isna().sum()}')
print()
print('Por ciudad:')
print(df_final['city'].value_counts())
print()
print('Precio medio / mediano / máximo por ciudad:')
print(df_final.groupby('city')['price'].agg(['mean','median','max']).round(1))
print()
print('Nulos restantes:')
nulos = df_final.isna().sum()
print(nulos[nulos > 0] if (nulos > 0).any() else '  Ninguno')

In [ ]:
# Comparación merged vs final — impacto de la limpieza global
print('Impacto de la limpieza global:')
print(f'  Shape merged: {df_merged.shape[0]:,} filas x {df_merged.shape[1]} cols  (más crudo, más columnas)')
print(f'  Shape final:  {df_final.shape[0]:,} filas x {df_final.shape[1]} cols  (limpio, con features globales)')
print(f'  Filas eliminadas en limpieza global: {df_merged.shape[0] - df_final.shape[0]:,} '
      f'({(df_merged.shape[0] - df_final.shape[0]) / df_merged.shape[0] * 100:.1f}%)')
print(f'  Columnas eliminadas/transformadas:   {df_merged.shape[1] - df_final.shape[1] + 5} '
      f'(netas: {df_final.shape[1] - df_merged.shape[1]} añadidas por features)')

In [ ]:
df_final.to_csv(f'{PROC}/listings_final.csv', index=False)
print('listings_final.csv guardado en data/processed/')
print()
print('Todos los archivos en data/processed/:')
for f in sorted(os.listdir(PROC)):
    kb = os.path.getsize(f'{PROC}/{f}') / 1024
    print(f'  {f:<38} {kb:>7.0f} KB')

## Tabla resumen de decisiones de limpieza

| Problema | Dónde | Decisión | Justificación |
|---|---|---|---|
| `price` como string `'$120.00'` | `*_clean` | Eliminar símbolo `$` y `,`, convertir a float | Necesario para operar numéricamente |
| Precio nulo o cero | `*_clean` | Filas eliminadas | Sin precio el registro no tiene utilidad analítica |
| Columnas de texto libre y URLs | `*_clean` | 15 columnas eliminadas | No aportan al análisis cuantitativo |
| `bathrooms`/`beds`/`bedrooms` nulos | `*_clean` | Imputados con mediana por ciudad | Mediana robusta frente a outliers |
| Booleanos como `'t'/'f'` | `*_clean` | Convertidos a `True/False` | Corrección de tipo para operar lógicamente |
| Fechas como string | `*_clean` | Convertidas a datetime | Necesario para calcular antigüedad |
| Barcelona excluida | — | Ciudad sin precio en scrape disponible | El scrape de dic. 2025 no contiene precios |
| Duplicados tras concat | `final` | `drop_duplicates()` | Pueden aparecer si un anuncio está en dos scrapes |
| Outliers precio | `final` | p99 por ciudad + cap 2.000€ Mallorca | Distribuciones estructuralmente diferentes entre ciudades; Mallorca con precios semanales mal etiquetados |
| `review_scores_*` nulos | `final` | Mediana global | Coherencia en la imputación entre todas las ciudades |
| `host_antiguedad_años` nulos (111) | Sin tratar | Se mantienen como `NaN` | Ausencia legítima de fecha de registro (0.17% del total) |
| Variables nuevas creadas | `final` | `tourism_pressure`, `is_popular`, `host_profesional`, `precio_categoria`, `host_antiguedad_años` | Requieren el dataset completo para percentiles y normalizaciones globales |